In [66]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,County,School Type,Alternative Education Accountability,Charter,Number of Students,% Economically Disadvantaged,% EB/EL Students,Overall Rating,Student Achievement Rating,School Progress Rating,Academic Growth Rating,Closing the Gaps Rating,Relative Performance Rating
0,1,MCLENNAN,Elementary,No,No,502,0.584,0.082,B,B,B,C,B,B
1,2,DALLAS,High School,No,No,447,0.718,0.233,A,A,A,B,A,A
2,3,DALLAS,Elementary,No,No,473,0.812,0.609,D,D,D,D,D,D
3,4,REFUGIO,Middle School,No,No,139,0.676,0.072,B,C,B,C,C,B
4,5,TARRANT,Middle School,No,No,793,0.916,0.327,C,F,C,C,C,F


In [67]:
subtask1 = len(df_test[(df_test.County == 'ANDERSON') & (df_test["School Type"] == "Elementary")])
subtask2 = df_test["Academic Growth Rating"].value_counts().index[0]

df_subtasks = pd.DataFrame({
    "subtaskID": [1, 2],
    "datapointID": [1, 2],
    "answer": [subtask1, subtask2]
})

df_subtasks

,subtaskID,datapointID,answer
0,1,1,1
1,2,2,C


In [68]:
df_train.columns

Index(['SampleID', 'County', 'School Type',
       'Alternative Education Accountability', 'Charter', 'Number of Students',
       '% Economically Disadvantaged', '% EB/EL Students', 'Overall Rating',
       'Student Achievement Rating', 'School Progress Rating',
       'Academic Growth Rating', 'Closing the Gaps Rating',
       'Relative Performance Rating'],
      dtype='object')

In [69]:
#simple preprocessing

label_enc = ['County', 'School Type']

from sklearn.preprocessing import LabelEncoder
for col in label_enc:
    enc = LabelEncoder()
    df_train[col] = enc.fit_transform(df_train[col])
    df_test[col] = enc.transform(df_test[col])

df_train['Alternative Education Accountability'] = df_train['Alternative Education Accountability'].map({"No": 0, "Yes": 1})
df_test['Alternative Education Accountability'] = df_test['Alternative Education Accountability'].map({"No": 0, "Yes": 1})

df_train['Charter'] = df_train['Charter'].map({"No": 0, "Yes": 1})
df_test['Charter'] = df_test['Charter'].map({"No": 0, "Yes": 1})

grades_dict = {"F": 0, "D": 1, "C": 2, "B": 3, "A": 4}
grades_cols = ['Overall Rating', 'Student Achievement Rating', 'School Progress Rating', 'Academic Growth Rating', 'Closing the Gaps Rating']

for col in grades_cols:
    df_train[col] = df_train[col].map(grades_dict)
    df_test[col] = df_test[col].map(grades_dict)

df_train.head()

,SampleID,County,School Type,Alternative Education Accountability,Charter,Number of Students,% Economically Disadvantaged,% EB/EL Students,Overall Rating,Student Achievement Rating,School Progress Rating,Academic Growth Rating,Closing the Gaps Rating,Relative Performance Rating
0,1,159,2,0,0,502,0.584,0.082,3,3,3,2,3,B
1,2,56,3,0,0,447,0.718,0.233,4,4,4,3,4,A
2,3,56,2,0,0,473,0.812,0.609,1,1,1,1,1,D
3,4,194,4,0,0,139,0.676,0.072,3,2,3,2,2,B
4,5,218,4,0,0,793,0.916,0.327,2,0,2,2,2,F


In [70]:
features = [col for col in df_test.columns if col != "SampleID"]
target = 'Relative Performance Rating'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

X.head()

,County,School Type,Alternative Education Accountability,Charter,Number of Students,% Economically Disadvantaged,% EB/EL Students,Overall Rating,Student Achievement Rating,School Progress Rating,Academic Growth Rating,Closing the Gaps Rating
0,159,2,0,0,502,0.584,0.082,3,3,3,2,3
1,56,3,0,0,447,0.718,0.233,4,4,4,3,4
2,56,2,0,0,473,0.812,0.609,1,1,1,1,1
3,194,4,0,0,139,0.676,0.072,3,2,3,2,2
4,218,4,0,0,793,0.916,0.327,2,0,2,2,2


In [89]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

model = LGBMClassifier(verbose = -1)

cross_val_score(model, X, y, cv=3, scoring='accuracy')

array([0.87697393, 0.87178545, 0.8806025 ])

In [90]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [91]:
df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.SampleID,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,3,8168,B
1,3,8169,C
2,3,8170,D
3,3,8171,C
4,3,8172,C


In [92]:
submission = pd.concat([df_subtasks, df_preds])

submission

,subtaskID,datapointID,answer
0,1,1,1
1,2,2,C
0,3,8168,B
1,3,8169,C
2,3,8170,D
...,...,...,...
864,3,9032,D
865,3,9033,B
866,3,9034,C
867,3,9035,A


In [93]:
submission.to_csv("submission.csv", index=False)